In [1]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

df = pd.DataFrame([
    ("March 2026 OBR forecast",  24),
    ("EY baseline forecast",     11),
    ("EY adverse scenario",      -7),
    ("OBR upside scenario",      37),
], columns=["scenario", "balance"])
df["neg"] = df["balance"] < 0
df["lab"] = df["balance"].map(lambda v: f"£{v}bn" if v >= 0 else f"−£{abs(v)}bn")

TEAL, RED = "#36B7B4", "#E6224B"

# dropdown: always show OBR March forecast + one chosen scenario
others = ["EY baseline forecast", "EY adverse scenario", "OBR upside scenario"]
dropdown = alt.binding_select(options=others, name="Compare against: ")
pick = alt.param(name="pick", value="EY baseline forecast", bind=dropdown)

base = alt.Chart(df).transform_filter(
    (alt.datum.scenario == "March 2026 OBR forecast") | (alt.datum.scenario == pick))
order = ["March 2026 OBR forecast", "EY baseline forecast",
         "EY adverse scenario", "OBR upside scenario"]
colour = alt.Color("neg:N", scale=alt.Scale(domain=[False, True], range=[TEAL, RED]), legend=None)

bars = base.mark_bar(cornerRadius=2, size=80).encode(
    x=alt.X("scenario:N", sort=order, title=None,
            axis=alt.Axis(labelAngle=0, labelFontSize=11, labelPadding=8,
                          domain=False, ticks=False, labelLimit=140)),
    y=alt.Y("balance:Q", title="Current budget balance, 2029-30 (£bn)",
            scale=alt.Scale(domain=[-15, 45]),
            axis=alt.Axis(grid=True, gridDash=[3, 3], gridColor="#d9dce3",
                          values=list(range(-15, 46, 10)))),
    color=colour)

lab_pos = base.transform_filter(alt.datum.balance >= 0).mark_text(
    baseline="bottom", dy=-6, fontSize=15, fontWeight="bold").encode(
    x=alt.X("scenario:N", sort=order), y="balance:Q", text="lab:N", color=colour)
lab_neg = base.transform_filter(alt.datum.balance < 0).mark_text(
    baseline="top", dy=6, fontSize=15, fontWeight="bold").encode(
    x=alt.X("scenario:N", sort=order), y="balance:Q", text="lab:N", color=colour)

zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#122b39", strokeWidth=1).encode(y="y:Q")

chart = (zero + bars + lab_pos + lab_neg).add_params(pick).properties(
    width=440, height=340).configure(
    background="white", font="Circular Std").configure_view(fill="transparent", stroke="transparent")

chart.save("budget_scenarios.html")   # interactive (dropdown works)
chart.save("budget_scenarios.json")   # Vega-Lite spec (interactive if rendered)
styles.save(chart, path=".", name="budget_scenarios", svg=True)  # static fallback
chart.save("budget_scenarios.png", scale_factor=2.0)
chart

alt.LayerChart(...)